In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ Библиотеки импортированы")

✅ Библиотеки импортированы


In [5]:
# Ячейка 2: Загрузка датасета
print("📥 Загрузка датасета...")
df = pd.read_parquet('../data/raw/cic-collection.parquet')
print(f"Загружено: {len(df):,} записей")

# Бинарная классификация: 0 = нормальный, 1 = атака
df['label'] = df['Label'].apply(lambda x: 0 if x in ['Benign', 'Normal'] else 1)
print(f"Атаки: {(df['label'] == 1).sum():,}")
print(f"Норма: {(df['label'] == 0).sum():,}")

📥 Загрузка датасета...
Загружено: 9,167,581 записей
Атаки: 1,981,392
Норма: 7,186,189


In [6]:
# Ячейка 3: Выбор признаков (ТОЧНЫЕ названия из вашего датасета)
import numpy as np

# Признаки, которые точно есть в вашем CIC-IDS-Collection
FEATURES = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',  # ← ВАЖНО: "Backward", не "Bwd"
    'Fwd Packet Length Mean',
    'Bwd Packet Length Mean',
    'Flow Bytes/s',
    'Flow Packets/s',
    'Fwd PSH Flags',
    'Avg Packet Size',
    'Fwd Act Data Packets',    # ← "Packets", не "Pkts"
    'Fwd Seg Size Min',
    'Active Mean',
    'Active Std',
    'Idle Mean',
    'Idle Std'
]

# Фильтруем только существующие колонки (защита от ошибок)
FEATURES = [f for f in FEATURES if f in df.columns]

print(f"✅ Используем {len(FEATURES)} признаков:")
for i, f in enumerate(FEATURES, 1):
    print(f"   {i:2d}. {f}")

# Очистка данных
df_clean = df.dropna(subset=FEATURES)
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna(subset=FEATURES)

print(f"\n✅ После очистки: {len(df_clean):,} записей")

✅ Используем 15 признаков:
    1. Flow Duration
    2. Total Fwd Packets
    3. Total Backward Packets
    4. Fwd Packet Length Mean
    5. Bwd Packet Length Mean
    6. Flow Bytes/s
    7. Flow Packets/s
    8. Fwd PSH Flags
    9. Avg Packet Size
   10. Fwd Act Data Packets
   11. Fwd Seg Size Min
   12. Active Mean
   13. Active Std
   14. Idle Mean
   15. Idle Std

✅ После очистки: 9,167,581 записей


In [7]:
# Ячейка 4: Подготовка данных для обучения
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

# Бинарная классификация: 0 = нормальный, 1 = атака
# Проверяем какие колонки есть для целевой переменной
if 'Label' in df_clean.columns:
    target_col = 'Label'
elif 'ClassLabel' in df_clean.columns:
    target_col = 'ClassLabel'
else:
    raise ValueError("Не найдена целевая колонка!")

print(f"🎯 Используем целевую колонку: {target_col}")

# Преобразование меток: нормальный трафик = 0, атака = 1
df_clean['label'] = df_clean[target_col].apply(
    lambda x: 0 if str(x).lower() in ['benign', 'normal', '0'] else 1
)

print(f"\n📊 Распределение классов:")
print(df_clean['label'].value_counts())

# Разделение на признаки и целевую переменную
X = df_clean[FEATURES].values
y = df_clean['label'].values

# Разделение на обучающую и тестовую выборки (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📈 Разделение данных:")
print(f"   Train: {len(X_train):,} записей")
print(f"   Test:  {len(X_test):,} записей")

# Нормализация признаков
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Сохранение артефактов для плагина
joblib.dump(scaler, '../data/models/scaler.pkl')
joblib.dump(FEATURES, '../data/models/features.pkl')

print("\n✅ Артефакты сохранены:")
print("   - ../data/models/scaler.pkl")
print("   - ../data/models/features.pkl")

🎯 Используем целевую колонку: Label

📊 Распределение классов:
0    7186189
1    1981392
Name: label, dtype: int64

📈 Разделение данных:
   Train: 7,334,064 записей
   Test:  1,833,517 записей

✅ Артефакты сохранены:
   - ../data/models/scaler.pkl
   - ../data/models/features.pkl


In [8]:
# Ячейка 5: Обучение нейросетевой модели
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Для воспроизводимости
tf.random.set_seed(42)

# Простая архитектура MLP
model = keras.Sequential([
    layers.Input(shape=(len(FEATURES),)),
    
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    
    layers.Dense(1, activation='sigmoid')  # бинарная классификация
], name='suricata_nn_simple')

# Компиляция
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("📋 Архитектура модели:")
model.summary()

# Обучение
print("\n🚀 Начало обучения...")
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=15,
    batch_size=2048,  # большой batch для скорости
    verbose=1
)

# Сохранение модели
model.save('../data/models/simple_model.h5')
print("\n✅ Модель сохранена: ../data/models/simple_model.h5")

📋 Архитектура модели:
Model: "suricata_nn_simple"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                1024      
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dropout_1 (Dropout)         (None, 32)                0         
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 3,137
Trainable params: 3,137
Non-trainable params: 0
_________________________________________________________________

🚀 Начало обучения...
Ep

In [9]:
# Ячейка 6: Оценка модели на тестовой выборке
from sklearn.metrics import classification_report, accuracy_score

# Предсказания
y_pred_proba = model.predict(X_test, verbose=0)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

# Метрики
print("\n📊 Результаты на тестовой выборке:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))


📊 Результаты на тестовой выборке:
Accuracy: 0.9643

Classification Report:
              precision    recall  f1-score   support

      Normal       0.97      0.98      0.98   1437238
      Attack       0.93      0.90      0.92    396279

    accuracy                           0.96   1833517
   macro avg       0.95      0.94      0.95   1833517
weighted avg       0.96      0.96      0.96   1833517



In [2]:
import json
import tensorflow as tf
import joblib
import numpy as np
from pathlib import Path

# Пути к модели и логам
MODEL_PATH = '../data/models/simple_model.h5'  # ← Добавили ../
SCALER_PATH = '../data/models/scaler.pkl'      # ← Добавили ../
FEATURES_PATH = '../data/models/features.pkl'  # ← Добавили ../
eve_path = r'\\wsl$\Ubuntu\var\log\suricata\eve.json' # или путь через WSL

print("🔄 Загрузка модели...")
model = tf.keras.models.load_model(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)
features = joblib.load(FEATURES_PATH)

print(f"✅ Модель загружена. Признаков: {len(features)}")

# Функция извлечения признаков из EVE JSON
def extract_features(event):
    feature_dict = {}
    
    if 'flow' in event:
        flow = event['flow']
        # Извлеките нужные признаки из flow
        # (адаптируйте под ваши FEATURES)
        feature_dict['Flow Duration'] = float(flow.get('duration', 0))
        feature_dict['Total Fwd Packets'] = flow.get('pkts_toserver', 0)
        feature_dict['Total Bwd Packets'] = flow.get('pkts_toclient', 0)
        # ... добавьте остальные признаки
    
    return [feature_dict.get(f, 0) for f in features]

# Анализ логов
print(f"\n📄 Анализ логов: {EVE_PATH}")
alerts = []

with open(EVE_PATH, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f, 1):
        try:
            event = json.loads(line.strip())
            
            # Пропускаем события stats
            if event.get('event_type') == 'stats':
                continue
            
            # Извлекаем признаки
            feature_vector = extract_features(event)
            
            # Нормализация
            features_scaled = scaler.transform([feature_vector])
            
            # Предсказание
            pred_proba = float(model.predict(features_scaled, verbose=0)[0][0])
            
            # Если атака
            if pred_proba > 0.5:
                alert = {
                    'timestamp': event.get('timestamp'),
                    'src_ip': event.get('src_ip'),
                    'dest_ip': event.get('dest_ip'),
                    'confidence': pred_proba,
                    'label': 'ATTACK'
                }
                alerts.append(alert)
                
                print(f"\n🚨 ALERT [{line_num}]")
                print(f"   {event.get('src_ip')} → {event.get('dest_ip')}")
                print(f"   Уверенность: {pred_proba:.2%}")
                
        except Exception as e:
            if line_num <= 5:  # Покажем первые 5 ошибок
                print(f"⚠️  Error line {line_num}: {e}")
            continue

print(f"\n📊 ИТОГИ:")
print(f"   Всего событий проанализировано: {line_num}")
print(f"   Обнаружено атак: {len(alerts)}")
print(f"   Процент атак: {len(alerts)/line_num*100:.2f}%")

# Сохранение алертов
if alerts:
    with open('results/nn_alerts.json', 'w', encoding='utf-8') as f:
        for alert in alerts:
            f.write(json.dumps(alert) + '\n')
    print(f"\n✅ Алерты сохранены: results/nn_alerts.json")

🔄 Загрузка модели...
✅ Модель загружена. Признаков: 15


NameError: name 'EVE_PATH' is not defined